# Convert NetCDF to Zarr with Dynamic BBox Subsetting

This notebook reads a NetCDF file, applies a robust spatial `bbox` subset, and writes a chunked Zarr store.

It is designed to work with datasets covering different spatial scopes (for example Africa-wide and Morocco-only files).

In [1]:
from pathlib import Path

import numpy as np
import xarray as xr

In [ ]:
# Update these values for your run
input_netcdf = Path("/path/to/input_file.nc")
output_zarr = Path("/path/to/output_store.zarr")

# Bounding box in (lon_min, lon_max, lat_min, lat_max)
# Example Morocco-ish bbox:
bbox = (-13.5, -0.5, 27.0, 36.5)

# Chunking profile oriented for spatial queries
# Tune based on your variable sizes and access pattern
target_chunks = {"time": 30, "lat": 200, "lon": 200}

In [ ]:
def detect_coord_name(ds, candidates):
    lower_map = {name.lower(): name for name in list(ds.coords) + list(ds.dims)}
    for candidate in candidates:
        if candidate in lower_map:
            return lower_map[candidate]
    return None


def normalize_lon_value(lon, convention):
    if convention == "0_360":
        return lon % 360
    return ((lon + 180) % 360) - 180


def infer_lon_convention(lon_values):
    lon_min = float(np.nanmin(lon_values))
    lon_max = float(np.nanmax(lon_values))
    if lon_min >= 0 and lon_max > 180:
        return "0_360"
    return "-180_180"


def ordered_slice(coord_values, vmin, vmax):
    ascending = bool(coord_values[0] <= coord_values[-1])
    return slice(vmin, vmax) if ascending else slice(vmax, vmin)


def subset_bbox(ds, bbox):
    lon_min, lon_max, lat_min, lat_max = bbox

    lat_name = detect_coord_name(ds, ["lat", "latitude", "y"])
    lon_name = detect_coord_name(ds, ["lon", "longitude", "x"])

    if lat_name is None or lon_name is None:
        raise ValueError("Could not detect latitude/longitude coordinates in dataset.")

    lat_values = np.asarray(ds[lat_name].values)
    lon_values = np.asarray(ds[lon_name].values)

    lon_convention = infer_lon_convention(lon_values)
    lon_min_n = normalize_lon_value(lon_min, lon_convention)
    lon_max_n = normalize_lon_value(lon_max, lon_convention)

    lat_data_min, lat_data_max = sorted([float(np.nanmin(lat_values)), float(np.nanmax(lat_values))])
    lon_data_min, lon_data_max = sorted([float(np.nanmin(lon_values)), float(np.nanmax(lon_values))])

    lat_min_i = max(min(lat_min, lat_max), lat_data_min)
    lat_max_i = min(max(lat_min, lat_max), lat_data_max)

    if lat_min_i > lat_max_i:
        raise ValueError("Requested latitude bbox does not intersect dataset extent.")

    lat_sel = ordered_slice(lat_values, lat_min_i, lat_max_i)
    ds_lat = ds.sel({lat_name: lat_sel})

    # Handle antimeridian crossing after normalization
    if lon_min_n <= lon_max_n:
        lon_min_i = max(lon_min_n, lon_data_min)
        lon_max_i = min(lon_max_n, lon_data_max)
        if lon_min_i > lon_max_i:
            raise ValueError("Requested longitude bbox does not intersect dataset extent.")
        lon_sel = ordered_slice(lon_values, lon_min_i, lon_max_i)
        out = ds_lat.sel({lon_name: lon_sel})
    else:
        part1_min = max(lon_min_n, lon_data_min)
        part1_max = lon_data_max
        part2_min = lon_data_min
        part2_max = min(lon_max_n, lon_data_max)

        chunks = []
        if part1_min <= part1_max:
            chunks.append(ds_lat.sel({lon_name: ordered_slice(lon_values, part1_min, part1_max)}))
        if part2_min <= part2_max:
            chunks.append(ds_lat.sel({lon_name: ordered_slice(lon_values, part2_min, part2_max)}))

        if not chunks:
            raise ValueError("Requested longitude bbox does not intersect dataset extent.")
        out = xr.concat(chunks, dim=lon_name)

    return out, lat_name, lon_name

In [ ]:
if not input_netcdf.exists():
    raise FileNotFoundError(f"Input file not found: {input_netcdf}")

# Open lazily for large files
ds = xr.open_dataset(input_netcdf, chunks="auto")

ds_subset, lat_name, lon_name = subset_bbox(ds, bbox)
print(f"Detected coordinates -> lat: {lat_name}, lon: {lon_name}")
ds_subset

In [ ]:
# Map generic chunk config to actual coordinate names in this file
chunks_for_ds = {}
for key, value in target_chunks.items():
    if key == "lat":
        chunks_for_ds[lat_name] = value
    elif key == "lon":
        chunks_for_ds[lon_name] = value
    elif key in ds_subset.dims:
        chunks_for_ds[key] = value

ds_chunked = ds_subset.chunk(chunks_for_ds) if chunks_for_ds else ds_subset
print(f"Chunks used: {chunks_for_ds}")

ds_chunked.to_zarr(output_zarr, mode="w", consolidated=True)
print(f"Zarr store written to: {output_zarr}")

In [ ]:
# Quick check: reopen the Zarr store
ds_zarr = xr.open_zarr(output_zarr, consolidated=True)
ds_zarr